# WSJ 2027 - Gruppindelning Direktresa

Assign confirmed direktresa deltagare into groups of exactly 36.

Direktresa participants travel independently to/from WSJ in Poland.
Same grouping algorithm as rundresa but applied to the direktresa subset.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — Hilbert curve sort
3. **Friend wish** — at least ONE friend in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex balance

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 1614 participants
Confirmed: 1545, Unconfirmed: 34, Cancelled: 35
Total confirmed participants: 1544
Skipped: 34 unconfirmed, 1 wrong age/no DOB

By category:
category
deltagare    1296
ist           226
cmt            22

By travel type:
travel
rundresa      1075
direktresa     290
egen_resa      157
other           22

Friend wishes:
  With friend 1 member no: 885
  With friend 2 member no: 526
  With friend 1 name (text): 73
  With friend 2 name (text): 49

Skipped participants:
  DELTAGARE wrong age: Fredrik Berg born 1979-04-16 (age 48)


In [2]:
GROUP_SIZE = 36

# Filter to direktresa deltagare only
df_direkt = df[df['travel'] == 'direktresa'].copy().reset_index(drop=True)
print(f"=== Direktresa deltagare ===")
print(f"Participants: {len(df_direkt)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_direkt)
df_sorted = u.add_hilbert_index(df_direkt)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Direktresa deltagare ===
Participants: 290
With coordinates: 263
Without coordinates (Sweden centroid): 27

Groups: 8 x 36 + 1 x 2 = 9 total

By region:
region
Region Stockholm    115
Region Södra         45
Region Norr/Mitt     43
Region Västra        43
Region Östra         28
                      1

By age:
age
14    83
15    83
16    75
17    49

By sex:
sex
Man       148
Kvinna    140
Annat       2


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 20
Matched & applied: 10
Generic wishes (not a person): 0
Unresolved (friend not in project): 10

Matched:
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via exact
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via fuzzy(0.92)+kar
  Armilde Westerblom -> Loke Jageteg (Falkenbergs Scoutkår) [rundresa] via exact
  Armilde Westerblom -> Vera Hedgren (Bohus Scoutkår) [rundresa] via exact
  Astrid Dodd -> Gustav Glimtoft (Älvsjö Scoutkår) [direktresa] via exact
  Elsa Mattsson -> Julia Gunnarsson (Vendelsö Scoutkår) [direktresa] via exact
  Freja Caro -> Karin Hugner (S:t Botvids Scoutkår) [direktresa] via exact
  Rasmus Noring -> Victor Thorburn (Hamre Scoutkår) [rundresa] via exact
  Vera Turesson -> Vilda Englund (Equmenia Scout) [direktresa] via exact
  Vilda Englund -> Vera Turesson (Equmenia Scout) [direktresa] via exact

Unresolved:
  Elsa Mattsson -> "Dejan Greve Vendelsö scoutkår" (no ma

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 290
Groups: 8 x 36 + 1 x 2 = 9 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 135/145
  Kar violations: 3
  Avg geo spread: 1.5002

=== Phase 2: Fix friend wishes ===
  Swaps: 7
  Friend satisfaction: 145/145
  Kar violations: 3
  Avg geo spread: 1.7609

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 3
  Kar violations: 0
  Friend satisfaction: 141/145
  Avg geo spread: 1.7611

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 3
  Friend satisfaction: 144/145
  Kar violations: 0
  Avg geo spread: 1.7611

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 459
  Diversity score: 24.46 -> 24.66
  Avg geo spread: 1.7611 -> 1.9375

=== FINAL RESULTS ===
Groups: 8 x 36 + 1 x 2
Friend satisfaction: 145/145 (100%)
Kar violations: 0
Total swaps: 472
Diversity: 24.66
Avg geo spread: 1.9375


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 13/13      4   15/20/1  11   8  11   6        43    19
    2      36 21/21      4   17/19/0  14   9   9   4        82    22
    3      36 19/19      7   20/16/0   9  12   9   6       196    19
    4      36 20/20      4   25/11/0  10  11   6   9       119    22
    5      36 19/19      5   14/21/1  11   7  12   6       110    21
    6      36 18/18      3   17/19/0  12  10   9   5       109    26
    7      36 13/13      5   20/16/0   9  14   7   6        70    25
    8      36 21/21      6   19/17/0   7  10  12   7        26    16
    9       2   1/1      2     1/1/0   0   2   0   0         0     1
--------------------------------------------------------------------------------
Avg geographic spread: 84 km
Min/Max spread: 0 / 196 km


In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_direktresa')

Saved 290 participants to /config/notebooks/wsj27/wsj27_direktresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_direktresa_grupper.json

CSV preview (first 10 rows):
 group             name member_no  age    sex                kar                   district       region friend_1 friend_2       lat       lng
     1    Nora Hultberg   3451529   16 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.664664 13.346159
     1    Amos Svensson   3378005   15    Man Djupadals Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.575069 12.957032
     1 Eleonora Henning   3355820   15 Kvinna Djupadals Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.575069 12.957032
     1      Jamie Arndt   3437949   16    Man Djupadals Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.575069 12.957032
     1     Miki Lessing   3371249   14  Annat Djupadals Scoutkår Södra Skånes Scoutdistrikt Region 

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_direktresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Direktresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 115 (111 satisfied, 4 unsatisfied)
Group arcs: 5041 connections across 9 groups
Saved group map: /config/notebooks/wsj27/wsj_direktresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_direktresa_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_direktresa_grupper.json
  Map:  /config/notebooks/wsj27/wsj_direktresa_karta.html
